# JOUR 701 · Block 3 | Chi-square test of independence
### Follow-along Google Colab Lab · Same Message Trust dataset

**Research question:** Is the likelihood of **clicking the article** related to which **message condition** participants saw?

- **Categorical variable 1:** `condition` = Control or Personalized.
- **Categorical variable 2:** `clicked_article` = Yes or No.
- **What you will read:** observed **counts**, **percentages** within each condition, **expected counts**, **χ²**, **degrees of freedom (df)** and **p-value**.

**We are not comparing mean trust scores today.** We are comparing *how many people* are in different categories.

You do **not** need to memorize code or calculate χ² by hand. Run the cells in order, then interpret the results in ordinary language.

> **Important:** The numbers in some lecture slides were **illustrative examples**. This notebook calculates the real results from the course CSV.

## 1 · Quick recap: which statistical tool?

| Question | Variable types | Analysis | Read first |
|---|---|---|---|
| Do two scores move together? | numeric + numeric | Pearson correlation | *r* and *p* |
| Do TWO groups have different mean scores? | 2-category group + numeric outcome | t-test | group means, SDs, *p* |
| Does a predictor help explain a numeric score? | numeric outcome + predictor | Linear regression | coefficient, CI/SE, *p*, R² |
| Do THREE+ groups have different mean scores? | 3+-category group + numeric outcome | ANOVA | group means, SDs, *F*, *p* |
| Are categories associated? | **categorical + categorical** | **Chi-square independence** | **counts, percentages, χ², p** |

**Reminder:** *p* < .05 is the conventional cutoff for statistical significance. A result does **not** have to be significant to be useful. *p* ≥ .05 means **fail to reject the null**, not “prove no association.”

## 2 · Load the SAME Block 3 dataset

The original course CSV lives in your course GitHub repository. You do not need to upload the CSV to Colab. Make sure Colab has internet access to load the GitHub URL.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency, fisher_exact

url = (
    "https://raw.githubusercontent.com/KayChansiri/"
    "Course_research-methods-in-the-age-of-ai/main/data/"
    "JOUR701_Block3_Course_Dataset.csv"
)

raw = pd.read_csv(url)
print("Rows in original CSV:", len(raw))
print("Repeated participant IDs:", raw.duplicated("participant_id").sum())
display(raw[["participant_id", "condition", "clicked_article"]].head())

## 3 · Clean ONLY the columns we need

**Why clean?** The original dataset contains at least one repeated participant ID and inconsistent capitalization of `condition`. Treating `personalized` and `Personalized` as two different categories would produce the wrong table.

- **One row = one participant.** Keep one record per ID.
- **Standardize category labels.** Turn `personalized` into `Personalized` and `yes` into `Yes` if needed.
- **Do not make up missing clicks.** Exclude missing or invalid values from this analysis.

We will keep only the expected categories, and the notebook will tell us how many observations remain.

In [ ]:
df = raw.drop_duplicates(subset="participant_id", keep="first").copy()

for col in ["condition", "clicked_article"]:
    df[col] = df[col].astype("string").str.strip().str.title()

valid_condition = ["Control", "Personalized"]
valid_click = ["No", "Yes"]

chi_df = df.loc[
    df["condition"].isin(valid_condition)
    & df["clicked_article"].isin(valid_click),
    ["participant_id", "condition", "clicked_article"]
].copy()

print("Unique participant IDs:", len(df))
print("Participants with valid condition + click response:", len(chi_df))
print("Rows not usable in this chi-square test:", len(df) - len(chi_df))
print("Conditions:", sorted(chi_df["condition"].unique().tolist()))
print("Click responses:", sorted(chi_df["clicked_article"].unique().tolist()))
display(chi_df.head())

**Quick checkpoint:** What are the two categorical variables? Why should you avoid counting one participant twice?

## 4 · Make a two-way table (contingency table)

Each cell counts participants in a particular **combination** of categories: e.g., Control + clicked Yes.

- **Rows** = experimental condition.
- **Columns** = clicked article (No / Yes).
- **Cell values** = number of participants.

Notice that the counts and percentages printed below come from **the actual CSV**, not the lecture illustration.

In [ ]:
counts = pd.crosstab(
    chi_df["condition"], chi_df["clicked_article"]
).reindex(index=valid_condition, columns=valid_click, fill_value=0)

if (counts.sum(axis=1) == 0).any() or (counts.sum(axis=0) == 0).any():
    raise ValueError("A row or column has no observations; inspect the labels before testing.")

display(counts)
print("Total participants in table:", int(counts.to_numpy().sum()))

**Say it aloud:** “Among people in the Control condition, ___ clicked and ___ did not click.” Repeat for Personalized.

## 5 · Convert counts to ROW percentages

Counts alone can mislead when group sizes differ. A **row percentage** tells you the share of each condition that clicked.

Calculate the percentage *inside each message condition* (not down each click column).

In [ ]:
row_pct = counts.div(counts.sum(axis=1), axis=0) * 100

display(row_pct.round(1).rename(columns={
    "No": "Did not click (%)", "Yes": "Clicked (%)"
}))

click_rate_control = float(row_pct.loc["Control", "Yes"])
click_rate_personalized = float(row_pct.loc["Personalized", "Yes"])
percentage_point_gap = click_rate_personalized - click_rate_control

print(f"Control click rate: {click_rate_control:.1f}%")
print(f"Personalized click rate: {click_rate_personalized:.1f}%")
print(f"Difference (Personalized - Control): {percentage_point_gap:+.1f} percentage points")

**Checkpoint:** Which group has the higher **sample** click rate? By how many **percentage points**? This describes the sample; it does not yet establish a statistically significant association.

## 6 · Picture the two conditions

Each bar adds up to **100%** of the people in a condition. Use the plot to see the difference in clicking rates.

In [ ]:
ax = row_pct[["No", "Yes"]].plot(
    kind="bar", stacked=True, figsize=(8, 4.5),
    color=["#C5D8E7", "#2E70B5"], width=0.58
)
ax.set_ylabel("Participants within each condition (%)")
ax.set_xlabel("Message condition")
ax.set_ylim(0, 100)
ax.set_xticklabels(["Control", "Personalized"], rotation=0)
ax.legend(["Did not click", "Clicked"], title="Article click")
ax.set_title("Did participants click the article? (actual course data)")
for i, group in enumerate(valid_condition):
    p_yes = float(row_pct.loc[group, "Yes"])
    ax.text(i, 100 - p_yes / 2, f"{p_yes:.1f}% clicked",
            ha="center", va="center", color="white", weight="bold")
plt.tight_layout()
plt.show()

## 7 · Write the hypotheses before the test

**H₀ (null):** In the population, message condition and article clicking are **independent**. The click rate does **not** differ between the two conditions.

**H₁ (alternative):** In the population, message condition and article clicking are **associated**. The click rate **does** differ between conditions.

**What does χ² compare?** The **observed** counts (the table you saw) to the **expected** counts we would anticipate if H₀ were true.

You will not calculate χ² manually. Python does this for you.


## 8 · NEW: How do we calculate expected counts?

**Observed count** = the number of participants we actually saw in a cell.

**Expected count** = the number we would expect in that cell **if H₀ were true** (no association between condition and clicking).

**Easy formula for one cell:**

> **Expected = (row total × column total) ÷ grand total**

**Illustration from our lecture slide (example numbers only):** Suppose there are 90 Control participants, 84 participants clicked Yes overall, and the grand total is 180. For **Control + Yes**:

> **(90 × 84) ÷ 180 = 42 expected clicks**

**Why?** If 84/180 = 46.7% of everyone clicked, then under H₀ we would expect about 46.7% of the Control group to click, too. 46.7% of 90 = 42.

The next code cell uses **your actual course dataset**. Its numbers may be different from the slide illustration. It calculates the expected count for Control + Yes and the complete expected-count table.

In [ ]:
# STEP A — Find the totals from the REAL observed-count table.
control_total = counts.loc["Control"].sum()       # All Control participants
clicked_yes_total = counts["Yes"].sum()          # All participants who clicked Yes
grand_total = counts.to_numpy().sum()              # All usable participants

# STEP B — One expected cell: Control + clicked Yes.
expected_control_yes = control_total * clicked_yes_total / grand_total

print("Control row total:", int(control_total))
print("Clicked Yes column total:", int(clicked_yes_total))
print("Grand total:", int(grand_total))
print(f"Expected Control + Yes = ({control_total} × {clicked_yes_total}) ÷ {grand_total}")
print(f"Expected Control + Yes = {expected_control_yes:.2f} participants")

# STEP C — Apply the same formula to ALL four cells.
row_totals = counts.sum(axis=1).to_numpy()
column_totals = counts.sum(axis=0).to_numpy()
manual_expected = pd.DataFrame(
    np.outer(row_totals, column_totals) / grand_total,
    index=counts.index,
    columns=counts.columns
)
print("\nOBSERVED counts (what actually happened):")
display(counts)
print("\nEXPECTED counts (what H₀ would predict):")
display(manual_expected.round(2))


**Quick checkpoint (say it aloud):** Why is the expected count **not necessarily a whole number**, even though it represents people? What would it mean if observed counts were almost the same as expected counts?

## 9 · Run Pearson’s chi-square test of independence

Now Python compares **observed** and **expected** counts across the entire table.

The code uses `correction=False` to show the **uncorrected Pearson chi-square** statistic introduced in class. For a 2 × 2 table, SciPy otherwise applies Yates’ correction by default, so your numerical result could differ if you omit this argument.

- **χ² statistic:** measures the discrepancy between observed and expected cell counts.
- **df:** degrees of freedom; for 2 × 2, df = 1.
- **p-value:** how surprising a χ² statistic at least this large would be if the two variables were independent.


In [ ]:
chi2, p, dof, expected = chi2_contingency(
    counts, correction=False
)
expected_counts = pd.DataFrame(
    expected, index=counts.index, columns=counts.columns
)

print(f"Pearson chi-square (χ²): {chi2:.3f}")
print("Degrees of freedom (df):", dof)
print("p-value:", f"{p:.4f}" if p >= 0.0001 else "< .0001")
print("\nExpected counts if condition and clicking are independent:")
display(expected_counts.round(2))
# Check: SciPy's expected counts should match the values we calculated.
assert np.allclose(expected_counts.to_numpy(), manual_expected.to_numpy())
print("Check passed: manual expected counts match SciPy's expected counts.")


**Checkpoint:** Which value do we compare with .05? Does this test compare numeric **means**, or category **counts**?

## 10 · Check whether the chi-square test is appropriate

The classical Pearson chi-square approximation works best when observations are independent and expected cell counts are sufficiently large. In this introductory class, **expected counts of at least 5 in every cell** is a useful rule of thumb.

If the expected counts are small in a **2 × 2 table**, Fisher’s exact test may be preferable. It answers a related question without relying on the chi-square approximation.

In [ ]:
smallest_expected = float(expected_counts.to_numpy().min())
print(f"Smallest expected count: {smallest_expected:.2f}")

if smallest_expected >= 5:
    print("Rule-of-thumb check: all expected cells are at least 5.")
else:
    print("Warning: an expected count is below 5. Interpret Pearson χ² cautiously.")
    odds_ratio, fisher_p = fisher_exact(counts)
    print("Fisher's exact p-value for this 2 x 2 table:", round(fisher_p, 4))

## 11 · Interpret the ACTUAL result in plain language

- **p < .05:** reject H₀. Evidence suggests clicking and condition are associated.
- **p ≥ .05:** fail to reject H₀. We do **not** have enough evidence of an association.
- In both cases, report the **click percentages**, because statistical significance alone does not tell readers the size or direction of the sample difference.

**Do not say:** “The p-value is the probability the null hypothesis is true.”

**Causality note:** A chi-square test on its own does not prove one variable caused another. In a properly randomized experiment, the design may support a causal interpretation of the assigned message condition under additional assumptions.

In [ ]:
def print_p(x):
    return f"{x:.4f}" if x >= 0.0001 else "< .0001"

if p < 0.05:
    finding = "There was statistically significant evidence of an association"
    decision = "REJECT H₀"
else:
    finding = "We did not find statistically significant evidence of an association"
    decision = "FAIL TO REJECT H₀"

print("Statistical decision:", decision)
print(
    f"{finding} between message condition and article clicking "
    f"in the sample, Pearson χ²({dof}) = {chi2:.2f}, p = {print_p(p)}. "
    f"The observed click rate was {click_rate_control:.1f}% for Control "
    f"and {click_rate_personalized:.1f}% for Personalized."
)

# Optional descriptive effect size. For a 2 x 2 table, Cramér's V = |phi|.
n = int(counts.to_numpy().sum())
cramers_v = np.sqrt(chi2 / (n * min(counts.shape[0]-1, counts.shape[1]-1)))
print(f"Cramér's V (optional association strength): {cramers_v:.3f}")

### Write your own response (2–3 sentences)

> We examined whether **[categorical variable #1]** was associated with **[categorical variable #2]** using a **chi-square test of independence**. **[Group A]** had a ___% click rate, compared with ___% for **[Group B]**. We **[rejected / failed to reject]** H₀, χ²(___) = ___, p = ___, because ____________________.

**Question:** Why would this wording be wrong? “The personalized group’s average article click was 54%.”

**Hint:** `clicked_article` is a category, so report the *percentage who clicked*, not an average on a numeric trust scale.

## 12 · Optional practice: a different 3 × 2 chi-square table

Change the first categorical variable to `education_level` (three groups), while the click outcome stays Yes / No. **We are still using chi-square**, NOT ANOVA, because the outcome is categorical.

This is exploratory observational subgroup analysis: do not say education level caused clicks.

In [ ]:
optional_df = df.loc[
    df["education_level"].notna()
    & df["clicked_article"].isin(valid_click),
    ["education_level", "clicked_article"]
].copy()
optional_df["education_level"] = optional_df["education_level"].astype(str).str.strip()

education_counts = pd.crosstab(
    optional_df["education_level"], optional_df["clicked_article"]
).reindex(columns=valid_click, fill_value=0)

display(education_counts)
if education_counts.shape[0] >= 2 and (education_counts.sum(axis=0) > 0).all():
    chi2_ed, p_ed, dof_ed, exp_ed = chi2_contingency(
        education_counts, correction=False
    )
    print(f"Education x clicking: χ²({dof_ed}) = {chi2_ed:.3f}, p = {print_p(p_ed)}")
    print("Minimum expected count:", round(float(exp_ed.min()), 2))
else:
    print("Not enough variation in category counts for this optional test.")

## 13 · Exit ticket (answer without coding)

1. What is the difference between the **numeric outcome** in ANOVA and the **categorical outcome** in chi-square?
2. What does the **contingency table** tell us?
3. What are **expected counts**?
4. What do you say if **p ≥ .05**?
5. Does a significant chi-square result by itself prove causation? Why or why not?

**Memory cue:**

- Numeric ↔ numeric relationship → **correlation**.
- Two groups + numeric outcome → **t-test**.
- Three+ groups + numeric outcome → **ANOVA**.
- Numeric outcome explained by predictors → **regression**.
- **Categorical + categorical** → **chi-square test of independence**.

You have learned to select a tool based on **research question → variable types → output interpretation**.